# 02_ex_agreement_topic — exploration and proposal

Analyst/data scientist notebook for source exploration, exploratory transforms, and proposal evidence.

> AI output in this notebook is advisory and requires human validation.
> Governance approval belongs to `01_data_sharing_agreement`; production enforcement belongs to `03_pc`.


## 01 Introduction and scope

Document:
- `agreement_id`
- `topic`
- source being explored
- question being answered
- approved usage context from the data sharing agreement


In [ ]:
topic = "REPLACE_TOPIC"
source_lakehouse_target = "source"
source_warehouse_target = "warehouse"
source_schema = "dbo"
source_table = "REPLACE_SOURCE_TABLE"
source_file_path = "REPLACE_SOURCE_FILE_PATH"
profile_output_table = "REPLACE_PROFILE_OUTPUT_TABLE"
proposal_output_table = "REPLACE_PROPOSAL_OUTPUT_TABLE"
metadata_evidence_table = "REPLACE_METADATA_EVIDENCE_TABLE"


## 02 Agreement context

Capture approved context from `01_data_sharing_agreement_<agreement>`:
- approved usage scope
- stakeholder owner/steward
- known restrictions relevant to this exploration


## 03 Configuration and setup


In [ ]:
%run 00_env_config


## 04 Data ingestion for exploration


In [ ]:
from fabricops_kit import (
    get_path,
    lakehouse_table_read,
    warehouse_read,
    profile_dataframe,
    setup_fabricops_notebook,
    load_agreements,
    select_agreement,
    get_selected_agreement,
    register_current_notebook,
)

env_name = ENV_NAME
metadata_path = get_path(env_name, "metadata", config=FABRIC_CONFIG)
table_name = source_table

runtime = setup_fabricops_notebook(
    config=FABRIC_CONFIG,
    env=env_name,
    required_targets=["source", "metadata"],
)

agreements = load_agreements(spark)
select_agreement(agreements)


## 05 Source profiling


In [ ]:
selected_agreement = get_selected_agreement()
agreement_id = selected_agreement["agreement_id"]
business_context = selected_agreement.get("business_context", "")
approved_usage = selected_agreement.get("approved_usage", "")
dataset_name = selected_agreement.get("agreement_name") or source_table

register_current_notebook(
    spark,
    metadata_path=metadata_path,
    agreement_id=agreement_id,
    notebook_type="02_ex",
    environment_name=env_name,
    dataset_name=dataset_name,
    table_name=table_name,
    topic=topic,
)

source_lakehouse = get_path(env_name, source_lakehouse_target, config=FABRIC_CONFIG)
source_df = lakehouse_table_read(source_lakehouse, source_table)

# Optional warehouse source alternative:
# source_df = warehouse_read(env=env_name, target=source_warehouse_target, schema=source_schema, table=source_table, config=FABRIC_CONFIG)

profile_df = profile_dataframe(source_df, table_name=source_table)
profile_df.show(20, truncate=False)


## 06 Exploratory analysis

Inspect distributions, nulls, duplicates, sample records, and capture analyst observations.


In [ ]:
# Example exploration snippets
source_df.selectExpr("count(*) as total_rows").show()
source_df.limit(20).show(truncate=False)

# TODO: add focused analyst observations for this topic.


## 07 Exploratory transforms

`02_ex` may contain exploratory transformation logic to test shaping ideas.

**Principle:** final repeatable run-all-safe transformation logic belongs in `03_pc`.


In [ ]:
# Optional exploratory transform placeholder
# ex_df = source_df.filter("some_column is not null")
# ex_df = ex_df.withColumn("derived_col", ...)
# ex_df.show(20, truncate=False)


## 08 AI-assisted DQ and classification proposals


In [ ]:
# Optional AI-assisted proposal placeholder
# from fabricops_kit import draft_dq_rules
# dq_candidates = draft_dq_rules(profile_df=profile_df, table_name=source_table)
# display(dq_candidates)

# NOTE: AI suggestions are advisory. Human validation and governance approval are required.


## 09 Findings and proposed metadata updates


In [ ]:
# Create a small proposal/evidence DataFrame if supported by your runtime
proposal_rows = [
    {"agreement_id": agreement_id, "topic": topic, "proposal_type": "dq_candidate", "rationale": "REPLACE"},
    {"agreement_id": agreement_id, "topic": topic, "proposal_type": "classification_candidate", "rationale": "REPLACE"},
]

# Spark-based creation pattern
proposal_df = spark.createDataFrame(proposal_rows)
proposal_df.show(truncate=False)

# Optional persistence
# proposal_df.write.mode("overwrite").saveAsTable(proposal_output_table)
# proposal_df.write.mode("append").saveAsTable(metadata_evidence_table)


## 10 Handoff to governance and pipeline contract

- Governance updates move to `01_data_sharing_agreement_<agreement>` for approval.
- Approved metadata/rules/classifications are consumed by `03_pc_<agreement>_<pipeline>`.
- Production transforms and enforcement happen in `03_pc`.


## 11 Frozen diagnostics / appendix

Keep one-time diagnostics and deep debug outputs here. Avoid turning exploratory cells into scheduled production behavior in `02_ex`.
